# 02. 데이터 전처리 & Feature Engineering

## 개요
EDA에서 발견한 패턴을 바탕으로 전처리 파이프라인을 구축한다.

## 전처리 전략
1. **MICE 결측치 보간** — 구조적 결측치를 다변량 방식으로 보간
2. **DI 시술 전용 처리** — IVF 전용 컬럼을 0으로 채움
3. **횟수 컬럼 수치화** — '0회', '1회' 등 문자열을 숫자로 변환

## Feature Engineering 전략
- **비율 피처**: 난자 성공률, 배아 사용률, 임신률/출산률
- **시기별 상호작용**: 시술 시기 × 배아 사용 여부 (가장 큰 성능 기여)
- **나이 기반 피처**: 난자 나이 계산, 나이 패널티
- **불임 원인 집계**: 남성/여성 불임 심각도
- **배아 품질 지표**: 배양 시간, 이상적 배양 기간 플래그

### 💡 실패한 시도와 배운 점
- **단순 0 대체**: 결측치를 0으로 채우면 DI 환자의 IVF 컬럼이 '0개 사용'으로 해석되어 노이즈 발생 → MICE로 전환
- **KNN Imputer**: MICE 대비 속도가 느리고 성능 차이 미미 → MICE 채택
- **PCA 차원 축소**: 범주형 변수가 많아 효과 미미 → 수작업 Feature Engineering이 더 효과적

---

## Import Libaries

In [4]:
! pip install miceforest

In [1]:
! pip install scikit-learn
! pip install catboost

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, MinMaxScaler, RobustScaler, StandardScaler
from sklearn.impute import KNNImputer
import warnings
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import random
import miceforest as mf
import sys
import os
from pathlib import Path
import joblib

warnings.filterwarnings('ignore')

## Data converting functions

### 횟수 컬럼 수치화
IVF/DI 시술 횟수, 임신 횟수 등이 '0회', '1회', '6회 이상' 등 문자열로 되어 있어 수치형으로 변환한다.
- '6회 이상' → 6으로 매핑 (상한 클리핑)

In [18]:
# 특정 column의 범주형 값을 수치형으로 변환하는 함수
def number_mapping(df):
    ivf_cols = [
        "IVF 임신 횟수", "IVF 출산 횟수", "DI 임신 횟수", "DI 출산 횟수",
        "IVF 시술 횟수", "총 임신 횟수", "DI 시술 횟수", "총 시술 횟수", "클리닉 내 총 시술 횟수","총 출산 횟수"
    ]

    mapping = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}

    for col in ivf_cols:
        if col in df.columns:  
            df[col] = df[col].replace(mapping) 
            df[col] = pd.to_numeric(df[col], errors='coerce')  


# 범주형, 수치형 columns들을 list에 추가 및 drop list에 있는 column drop 함수 
def update_column_list(df, category_columns, num_columns, drop_list):
    """
    전역 리스트(category_columns, num_columns)에 없는 컬럼을 dtype에 따라 자동으로 추가하는 함수.

    :param df: pandas DataFrame
    :result: 업데이트된 category_columns, num_columns list
    """
    
    # 기존 리스트 복사
    category_columns[:] = [col for col in category_columns if col not in drop_list]
    num_columns[:] = [col for col in num_columns if col not in drop_list]

    # df의 모든 컬럼을 검사하여 없던 컬럼을 자동으로 추가
    for col in df.columns:
        if col not in category_columns and col not in num_columns and col not in drop_list:
            if pd.api.types.is_numeric_dtype(df[col]):  # 숫자형 컬럼인지 확인
                num_columns.append(col)
            elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):  # 문자형인지 확인
                category_columns.append(col)
                
    df.drop(columns=drop_list, errors='ignore', inplace = True)


# Column 타입을 지정된 타입으로 변환하는 함수
def convert_type(df, category_columns, num_columns, drop_list):
    """
    컬럼 타입을 자동 변환하는 함수.
    """

    sub_category_columns = category_columns.copy()
    sub_num_columns = num_columns.copy()
    
    # _list에 있는 column들은 변환 리스트에서 제외
    sub_category_columns = [x for x in category_columns if x not in drop_list]
    sub_num_columns = [x for x in num_columns if x not in drop_list]

    # 컬럼 타입 변환
    for col in df.columns:
        if col in sub_category_columns:
            # 기존 dtype이 category가 아닌 경우 category로 변환
            if not pd.api.types.is_categorical_dtype(df[col]):
                df[col] = df[col].astype(str).astype('category')
                # sub_num_columns.append(col)  # category가 아니었으므로 숫자 컬럼 리스트에 추가

        elif col in sub_num_columns:
            # 기존 dtype이 numeric이 아닌 경우 float로 변환
            if not pd.api.types.is_numeric_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], errors='coerce')
                # sub_category_columns.append(col)  # 숫자가 아니었으므로 category 컬럼 리스트에 추가


# Catboost를 위해 category 함수르 string으로 변환
def convert_category_columns(df, category_columns):
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype(str)  # 모든 값을 문자열로 변환

## Data preprocess functions

### 횟수 컬럼 수치화
IVF/DI 시술 횟수, 임신 횟수 등이 '0회', '1회', '6회 이상' 등 문자열로 되어 있어 수치형으로 변환한다.
- '6회 이상' → 6으로 매핑 (상한 클리핑)

### 비율 피처 생성
원본 수치 컬럼들의 비율을 계산하여 새로운 피처를 생성한다.
- 분모가 0인 경우 -1로 처리하여 '해당 없음'을 표현
- 예: `난자 성공률 = 미세주입된 난자 수 / 수집된 신선 난자 수`

> 💡 비율 피처는 절대값보다 시술 효율을 더 잘 반영하여 성능 향상에 기여했다.

### 시기별 상호작용 피처 (핵심 피처)
시술 시기 코드와 배아 사용 여부를 결합한 상호작용 피처를 생성한다.
- 같은 시기라도 신선/동결/기증 배아 사용 여부에 따라 성공률이 크게 다름
- 이 피처가 **Feature Importance 상위권**에 위치하며 가장 큰 성능 기여

> 💡 단순 시술 시기 코드만으로는 포착할 수 없는 패턴을 상호작용으로 포착

### 난자 나이 계산
난자 출처(본인/기증)에 따라 실제 난자의 나이를 다르게 계산한다.
- 본인 제공 → 시술 당시 나이 사용
- 기증 제공 → 기증자 나이 사용 (없으면 젊은 나이로 대체)

> 💡 기증 난자는 일반적으로 젊은 여성에게서 제공되므로, 시술 당시 나이보다 난자 나이가 더 정확한 예측 변수

In [19]:
# Mice model이 결측치를 채우기 전 전처리
def pre_mice(df):
    df["배아 해동 여부"] = df["배아 해동 경과일"].notna().astype(int)
    df["배란 유도_배란 자극"] = df["배란 자극 여부"].astype(str) + "_" + df["배란 유도 유형"].astype(str)


# 시술 유형이 DI인 경우, 특정 column들 결측치 처리 함수
def pp_fillna_di(df):
    di_nan_col = [
        '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
        '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
        '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
        '기증자 정자와 혼합된 난자 수', "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
        "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "단일 배아 이식 여부", '난자 혼합 경과일',
        '임신 시도 또는 마지막 임신 경과 연수', 
    ]

    row = df['시술 유형'] == "DI"
    
    df.loc[row, di_nan_col] = df.loc[row, di_nan_col].fillna(0)


# 비율 및 수치 관련 전처리 함수
def pp_ratio(df):
    df['난자 성공률'] = np.where(df['수집된 신선 난자 수'] == 0, -1, df['미세주입된 난자 수'] / df['수집된 신선 난자 수'])
    df['저장된 배아 사용률'] = np.where(df['총 생성 배아 수'] + df['해동된 배아 수'] == 0, -1, df['저장된 배아 수'] / (df['총 생성 배아 수'] + df['해동된 배아 수']))

    # 시술 횟수 관련 전처리
    df['IVF 임신률'] = np.where(df['IVF 시술 횟수'] == 0, -1, df['IVF 임신 횟수'] / df['IVF 시술 횟수'])
    df['DI 임신률'] = np.where(df['DI 시술 횟수'] == 0, -1, df['DI 임신 횟수'] / df['DI 시술 횟수'])

    df['IVF 출산률'] = np.where(df['IVF 임신 횟수'] == 0, -1, df['IVF 출산 횟수'] / df['IVF 임신 횟수'])
    df['DI 출산률'] = np.where(df['DI 임신 횟수'] == 0, -1, df['DI 출산 횟수'] / df['DI 임신 횟수'])

    df['IVF 시술 비율'] = np.where((df['IVF 시술 횟수'] + df['DI 시술 횟수']) == 0, -1, 
                              df['IVF 시술 횟수'] / (df['IVF 시술 횟수'] + df['DI 시술 횟수']))

    df['총 출산률'] = np.where(df['총 임신 횟수'] == 0, -1, df['총 출산 횟수'] / df['총 임신 횟수'])

    df["클리닉 시술 비율"] = np.where((df["IVF 시술 횟수"] + df["DI 시술 횟수"]) == 0, -1, 
                                df["클리닉 내 총 시술 횟수"] / (df["IVF 시술 횟수"] + df["DI 시술 횟수"]))
    
    df['미세주입 배아 이식률'] = np.where((df['해동된 배아 수'] + df['미세주입에서 생성된 배아 수'] - df['미세주입 후 저장된 배아 수']) == 0, -1, 
                                      df['미세주입 배아 이식 수'] / (df['해동된 배아 수'] + df['미세주입에서 생성된 배아 수'] - df['미세주입 후 저장된 배아 수']))

    df['이식된 배아 수 대비 미세주입 배아 수'] = np.where(df['이식된 배아 수'] == 0, -1, 
                                                df['미세주입 배아 이식 수'] / df['이식된 배아 수'])

    df['미세주입 배아 생성률'] = np.where(df['미세주입된 난자 수'] == 0, -1, 
                                  df['미세주입에서 생성된 배아 수'] / df['미세주입된 난자 수'])

    df['혼합 대비 생성 배아률'] = np.where((df['혼합된 난자 수'] + df['해동 난자 수']) == 0, -1, 
                                    df['총 생성 배아 수'] / (df['혼합된 난자 수'] + df['해동 난자 수']))

    df['신선 난자 선택'] = np.where((df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']) == 0, -1, 
                            df['혼합된 난자 수'] / (df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']))

    df['수집 난자 이식률'] = np.where((df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']) == 0, -1, 
                                df['이식된 배아 수'] / (df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']))

    df['혼합 대비 미세주입 난자 비율'] = np.where(df['혼합된 난자 수'] == 0, -1, 
                                          df['미세주입된 난자 수'] / df['혼합된 난자 수'])

    df['경과일 대비 배아 이식 수'] = np.where(df['배아 이식 경과일'] == 0, -1, 
                                df['이식된 배아 수'] / df['배아 이식 경과일'])

    df['총 배아 사용 대비 이식'] = np.where((df['해동된 배아 수'] + df['총 생성 배아 수'] + df['저장된 배아 수']) == 0, -1, 
                                            df['이식된 배아 수'] / (df['해동된 배아 수'] + df['총 생성 배아 수'] + df['저장된 배아 수']))

    df['총 사용 배아'] = df['해동된 배아 수'] + df['총 생성 배아 수']
    df["IVF 정자 미혼합 여부"] = ((df["파트너 정자와 혼합된 난자 수"] == 0) & (df["기증자 정자와 혼합된 난자 수"] == 0) & (df["시술 유형"] == "IVF")).astype(int)
    
    df['대리모 여부'] = df['대리모 여부'].fillna(-1)
    df['배란 유도 유형'].replace({'생식선 자극 호르몬' : '기록되지 않은 시행', '세트로타이드 (억제제)': '기록되지 않은 시행'} , inplace = True)

    # 시술 시기에 따른 각 column들의 영향
    df['시기별 배아 사용 여부'] = (
            df['시술 시기 코드'].str.lower()
            + df['동결 배아 사용 여부'].fillna(0).astype(int).astype(str)
            + df['신선 배아 사용 여부'].fillna(0).astype(int).astype(str)
            + df['기증 배아 사용 여부'].fillna(0).astype(int).astype(str)
        ).astype('category')
    
    # 신선 배아 사용시 시기에 따른 난자 개수의 영향 반영
    df['과배란 유도 주사 효과'] = 'frozen'
    fresh = (df['신선 배아 사용 여부'] == 1) & (df['동결 배아 사용 여부'] == 0)

    mask2 = (df['수집된 신선 난자 수'] > 10) & fresh
    mask3 = (df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 10) & fresh

    df.loc[mask2, '과배란 유도 주사 효과'] = df.loc[mask2, '시술 시기 코드'] + '_1'
    df.loc[mask3, '과배란 유도 주사 효과'] = df.loc[mask3, '시술 시기 코드'] + '_2'

    # 동결 배아 사용시에 시기별 해동 배아의 수 반영 
    only_frozen = (df['동결 배아 사용 여부'] == 1) &  (df['신선 배아 사용 여부'] == 0)

    df['시기별 해동 배아 영향'] = 'frozen'

    mask2 = (df['해동된 배아 수'] > 3) & only_frozen
    mask3 = (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 3) & only_frozen

    df.loc[mask2, '시기별 해동 배아 영향'] = df.loc[mask2, '시술 시기 코드'] + '_1'
    df.loc[mask3, '시기별 해동 배아 영향'] = df.loc[mask3, '시술 시기 코드'] + '_2'

    # 신선/동결 배아 사용 시 시기별 난자 및 배아의 수 반영 
    df['시기별 배아 영향'] = 'both'
    both = (df['신선 배아 사용 여부'] == 1) & (df['동결 배아 사용 여부'] == 1)

    mask2 = ((df['수집된 신선 난자 수'] > 7) & (df['해동된 배아 수'] > 2)) & both
    mask3 = ((df['수집된 신선 난자 수'] > 7) & (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 2)) & both
    mask4 = ((df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 7) & (df['해동된 배아 수'] > 2)) & both
    mask5 = ((df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 7) & (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 2)) & both
    mask6 = (df['수집된 신선 난자 수'] == 0) & (df['해동된 배아 수'] > 2) & both
    mask7 = (df['수집된 신선 난자 수'] == 0) & (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 2) & both
    mask8 = (df['수집된 신선 난자 수'] > 7) & (df['해동된 배아 수'] == 0) & both
    mask9 = ((df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 7)) & (df['해동된 배아 수'] == 0) & both

    df.loc[mask2, '시기별 배아 영향'] = df.loc[mask2, '시술 시기 코드'] + '_1'
    df.loc[mask3, '시기별 배아 영향'] = df.loc[mask3, '시술 시기 코드'] + '_2'
    df.loc[mask4, '시기별 배아 영향'] = df.loc[mask4, '시술 시기 코드'] + '_3'
    df.loc[mask5, '시기별 배아 영향'] = df.loc[mask5, '시술 시기 코드'] + '_4'
    df.loc[mask6, '시기별 배아 영향'] = df.loc[mask6, '시술 시기 코드'] + '_5'
    df.loc[mask7, '시기별 배아 영향'] = df.loc[mask7, '시술 시기 코드'] + '_6'
    df.loc[mask8, '시기별 배아 영향'] = df.loc[mask8, '시술 시기 코드'] + '_7'
    df.loc[mask9, '시기별 배아 영향'] = df.loc[mask9, '시술 시기 코드'] + '_8'

    # 이상적인 배양 기간 설정
    mask = (df['배아 이식 경과일'].isin([3, 5]))
    df['이상적인 배양 기간'] = 'NO'
    df.loc[mask, '이상적인 배양 기간'] = 'YES'

    # 시기별 각 column들의 영향 정리
    df['시기별 단일 배아 이식 여부'] = (df['시술 시기 코드'].str.lower() + df['단일 배아 이식 여부'].astype(int).astype(str))
    df['시기별 유전 진단 사용 여부'] = df['시술 시기 코드'] + df['착상 전 유전 진단 사용 여부'].fillna(0).astype(int).astype(str)
    df['시기별 배란 자극 사용 여부'] = df['시술 시기 코드'] + df['배란 자극 여부'].fillna(0).astype(int).astype(str)
    
    # 신선 배아의 배양 시간에 따른 영향
    if all(col in df.columns for col in ["배아 이식 경과일", "난자 혼합 경과일"]):
        df["신선 배아 배양 시간"] = df["배아 이식 경과일"] - df["난자 혼합 경과일"]
        
    df["신선 배아 배양 시간"] = df["신선 배아 배양 시간"].fillna(0)

    # 동결 배아의 배양 시간에 따른 영향 
    if all(col in df.columns for col in ["배아 이식 경과일", "배아 해동 경과일"]):
        df["동결 배아 배양 시간"] = df["배아 이식 경과일"] - df["배아 해동 경과일"]
        
    df["동결 배아 배양 시간"] = df["동결 배아 배양 시간"].fillna(0)


# 시술 당시 나이가 알 수 없음인 경우 만45-50세로 대체
def pp_age_replace(df): 
    age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']
    df['시술 당시 나이'] = df['시술 당시 나이'].replace('알 수 없음', '만45-50세')


# PGD, PGS의 영향 전처리 함수수
def pp_merge_pgd_pgs(df):
    col = ["착상 전 유전 검사 사용 여부", "착상 전 유전 진단 사용 여부", "PGD 시술 여부", "PGS 시술 여부"]
    for i in col:
        df[col] = df[col].fillna(0)
    
    df["과거 유전자 검사 사용 여부"] = df["착상 전 유전 검사 사용 여부"] + df["착상 전 유전 진단 사용 여부"]
    df["현재 검사 사용 여부"] = df["PGD 시술 여부"] + df["PGS 시술 여부"]

    df["과거 PGD 비율"] = np.where(df["과거 유전자 검사 사용 여부"] == 0, -1, df["착상 전 유전 검사 사용 여부"] / df["과거 유전자 검사 사용 여부"])
    df["현재 PGD 비율"] = np.where(df["현재 검사 사용 여부"] == 0, -1, df["PGD 시술 여부"] / df["현재 검사 사용 여부"])
    
    for i in col:
        df[i] = df[i].fillna(0)
        df[i] = df[i].astype('category')


# 각 기증자 나이의 알 수 없음 전처리
def pp_donate_egg_sperm(df): # 임의 
    col = ["난자 기증자 나이", "정자 기증자 나이"]
    
    for i in col:
        df[i].replace({'알 수 없음' : '만 21-25세'} , inplace = True)


# 배아 생성 주요 이유의 값 정리
def pp_categorize_embryo_creation_reason(df):
    col= "배아 생성 주요 이유"
    df[col] = df[col].fillna("시술용")  # NaN은 기본적으로 '시술용'으로 설정

    category_map = {
        "현재 시술용": "시술용",
        "배아 저장용, 현재 시술용": "시술용",
        "난자 저장용, 현재 시술용" : "시술용",
        "현재 시술용" : "시술용",

        "기증용": "기증용",
        "기증용, 현재 시술용": "기증용",
        "기증용, 배아 저장용": "기증용",
        "기증용, 난자 저장용": "기증용",

        "배아 저장용": "배아 저장용",

        "난자 저장용": "난자 저장용",
        "난자 저장용, 배아 저장용": "난자 저장용",

        "난자 저장용, 배아 저장용, 연구용": "연구용",
        "연구용, 현재 시술용": "연구용",

        "기증용, 배아 저장용, 현재 시술용": "복합용도"
    }

    df[col] = df[col].replace(category_map)


# 난자의 나이를 계산하는 함수
def pp_egg_age(df):
    df["나이"] = "알 수 없음"

    # 본인 제공일 경우 시술 당시 나이 사용
    df.loc[df["난자 출처"] == "본인 제공", "나이"] = df.loc[df["난자 출처"] == "본인 제공", "시술 당시 나이"]
    
    # 기증 제공 & 난자 기증자 나이 정보가 있는 경우
    mask = (df["난자 출처"] == "기증 제공") & (df["난자 기증자 나이"] != "알 수 없음")
    df.loc[mask, "나이"] = df.loc[mask, "난자 기증자 나이"]

    # 기증 제공 & 난자 기증자 나이 정보가 없다면 젊은 사람으로 넣자
    mask = (df["난자 출처"] == "기증 제공") & (df["난자 기증자 나이"] == "알 수 없음")
    df.loc[mask, "나이"] = "만18-34세"
    
    mask = (df["난자 출처"] == "알 수 없음")
    df.loc[mask, "나이"] = df.loc[mask, "시술 당시 나이"]


# 불임 원인 colunms 전처리 함수
def pp_merge_fail_causes(df):
    male_fail = ["불임 원인 - 남성 요인", "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인",
                "불임 원인 - 정자 운동성", "불임 원인 - 정자 형태"]
    
    female_fail = ["불임 원인 - 난관 질환", "불임 원인 - 배란 장애", "불임 원인 - 여성 요인", 
              "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증"]
    
    df["남성 불임 심각도"] = df[male_fail].sum(axis = 1)
    df["여성 불임 심각도"] = df[female_fail].sum(axis = 1)
    
    df["여성 심각도"] = np.where((df['남성 불임 심각도'] + df["여성 불임 심각도"]) == 0, -1, df["여성 불임 심각도"] / (df['남성 불임 심각도'] + df["여성 불임 심각도"]))


# 모든 전처리 함수를 처리하는 통합 전처리 함수
def preprocess(train, test, category_columns, num_columns, drop_list, mdl):    
    mice_col_lst = [
        '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
        '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
        '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
        '기증자 정자와 혼합된 난자 수', "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
        "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "단일 배아 이식 여부", '난자 혼합 경과일',
        '임신 시도 또는 마지막 임신 경과 연수'
    ]

    # Mice 전 전처리 함수 및 시술 유형 DI 관련 결측치 리리
    for df in [train, test]:
        pre_mice(df)
        pp_fillna_di(df)
    
    # Mice : mice_col_list 중 결측치 채우기
    train, test = mice_process(train, test, mice_col_lst, mdl)

    for df in [train, test]:
        # 횟수(IVF) 문자열로 되어있는 것을 number로 mapping
        number_mapping(df)

        # 배아 주요 생성 이유를 Mapping
        pp_categorize_embryo_creation_reason(df)

        # 비율 전처리  
        pp_ratio(df)
        
        # 시술 당시 나이를 만45-50세 대체.
        pp_age_replace(df)

        # PGS, PGD 여부를 merge
        pp_merge_pgd_pgs(df)

        # 난자와 정자 기증자 나이 대체. 
        pp_donate_egg_sperm(df)

        # 난자 나이 계산
        pp_egg_age(df)

        # 불임 원인 전처리 
        pp_merge_fail_causes(df)

    # number, category column에 따라 Dataframe 변환
    for df in [train, test]:
        convert_type(df, category_columns, num_columns, drop_list)
        update_column_list(df, category_columns, num_columns, drop_list)
        convert_category_columns(df, category_columns)
        # CatBoost NaN 방지: 범주형은 'Unknown', 수치형은 -1
        for col in df.columns:
            if col in category_columns:
                df[col] = df[col].fillna('Unknown').astype(str)
            elif pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(-1)

    return train, test

## Mice model training and apply

In [20]:
# Mice model training 함수
def training_mice(df, target_columns):
    df_subset = df[target_columns].copy()
    
    mdl = mf.ImputationKernel(df_subset, random_state=42)
    mdl.mice(iterations=5, n_estimators=50, n_jobs=-1)

    return mdl

# Mice model을 train 또는 test 데이터 셋에 적용하는 함수
def apply_mice(df, target_columns, mdl, isTraining):
    df_subset = df[target_columns].copy()

    if isTraining:
        # Mice model로부터 train 데이터 결측치 추정
        imputed_df =  mdl.complete_data()
        print('Training dataset nan filled')
    else:
        # Train data로 학습된 Mice model로부터 test 데이터 결측치 생성  
        imputed_df = mdl.impute_new_data(df_subset).complete_data()
        print('Test dataset nan filled')

    df[target_columns] = imputed_df[target_columns]

    return df

# Mice 모델을 train, test에 적용하는 함수. 
def mice_process(train, test, target_columns, mdl):
    print("MICE list : ", target_columns)
    
    train = apply_mice(train, target_columns, mdl, True)
    test = apply_mice(test, target_columns, mdl, False)

    return train, test

In [21]:
train = pd.read_csv('../data/train.csv')

# Mice로 채우고자 하는 column list
target_columns = [
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
    '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
    '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
    '기증자 정자와 혼합된 난자 수', "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
    "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "단일 배아 이식 여부", '난자 혼합 경과일',
    '임신 시도 또는 마지막 임신 경과 연수'
]

pp_fillna_di(train)

'''
mdl = training_mice(train, target_columns)

# Mice model 저장
joblib.dump(mdl, "./kds_model.pkl")
print("mice 모델 학습 후 저장 완료")
''' #학습용 코드

mdl = joblib.load("./kds_model.pkl") # mice 로드
print("mice 모델 로드 완료")


mice 모델 로드 완료


## Load dataset and Mice model

In [22]:
# 범주형 및 수치형 column 정의
category_columns = [
    "시술 당시 나이", "총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수","DI 시술 횟수", "총 임신 횟수", "IVF 임신 횟수",  "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수", "시술 시기 코드", "시술 유형", "특정 시술 유형", "배란 자극 여부",  "배란 유도 유형", "단일 배아 이식 여부",
    "착상 전 유전 검사 사용 여부", "착상 전 유전 진단 사용 여부", "남성 주 불임 원인", "남성 부 불임 원인", "여성 주 불임 원인", "여성 부 불임 원인",
    "부부 주 불임 원인","부부 부 불임 원인","불명확 불임 원인", '불임 원인 - 여성 요인',  "불임 원인 - 난관 질환", "불임 원인 - 남성 요인", "불임 원인 - 배란 장애", "불임 원인 - 여성 요인",
    "불임 원인 - 자궁경부 문제",  "불임 원인 - 자궁내막증",  "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인",  "불임 원인 - 정자 운동성", 
    "불임 원인 - 정자 형태",  "배아 생성 주요 이유","난자 출처", "정자 출처", "난자 기증자 나이",  "정자 기증자 나이",
    "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "대리모 여부", "PGD 시술 여부", "PGS 시술 여부"
]

num_columns = [
    '임신 시도 또는 마지막 임신 경과 연수', "난자 채취 경과일",  "난자 해동 경과일",  "난자 혼합 경과일", "배아 이식 경과일", "배아 해동 경과일",
    "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수", "이식된 배아 수", "미세주입 배아 이식 수", "저장된 배아 수",
    "미세주입 후 저장된 배아 수", "해동된 배아 수", "해동 난자 수", "수집된 신선 난자 수", "저장된 신선 난자 수", "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",
]

# Drop columns list
drop_list = ['ID','불임 원인 - 여성 요인' ,'불임 원인 - 정자 면역학적 요인']

# Data & Mice model load
df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')
mice_mdl = joblib.load("./kds_model.pkl")

# Preprocess
df_train, df_test = preprocess(df_train, df_test, category_columns, num_columns, drop_list, mice_mdl)

MICE list :  ['미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수', '난자 채취 경과일', '난자 해동 경과일', '배아 이식 경과일', '배아 해동 경과일', '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부', '단일 배아 이식 여부', '난자 혼합 경과일', '임신 시도 또는 마지막 임신 경과 연수']
Training dataset nan filled
Test dataset nan filled


## Training CatBoost model

In [23]:
# K fold n_split 수 및 random state
k_fold = 25
random_state = 2024

# Optuna를 통해 얻은 hyperparameter
params = {
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'iterations': 3000,
    'depth': 7,
    'learning_rate': 0.059442066023287284,
    'l2_leaf_reg': 5.916099093406194,
    'bagging_temperature': 0.0021443688183697563,
    'random_strength': 0.989238115265615,
    'od_type': 'Iter',
    'od_wait': 200,
    'random_seed': random_state,
    'use_best_model': True,
    'class_weights': {0: 1.0, 1: 1.004614669263528},

    # 🔥 CPU 설정
    'task_type': 'CPU',
    'thread_count': -1,   # 모든 코어 사용
    'verbose': 100
}

# Evaluation metrics 계산을 위한 함수
def add_eval_metrics(eval_metrics, y_test, pred, pred_proba):
    eval_metrics['confusion'].append(confusion_matrix(y_test, pred))
    eval_metrics['accuracy'].append(accuracy_score(y_test, pred))
    eval_metrics['precision'].append(precision_score(y_test, pred))
    eval_metrics['recall'].append(recall_score(y_test, pred))
    eval_metrics['f1_score'].append(f1_score(y_test, pred))
    eval_metrics['roc_auc'].append(roc_auc_score(y_test, pred_proba))

# 각 Fold별 test 데이터 prediction list
test_pred_lst = []

# 각 평가 지표를 담고 있는 dictionary
eval_metrics = {
    'confusion' : [],
    'accuracy' : [],
    'precision' : [],
    'recall' : [],
    'f1_score' : [],
    'roc_auc' : []
}

# Stratified K-Fold : Target column의 비율을 동일하게 사용하기 위함. 
skf = StratifiedKFold(n_splits=k_fold, shuffle=True, random_state=random_state)

# Training start
for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train["임신 성공 여부"])):
    print(f"{fold+1} start")

    # Split train, validation dataset
    train_data, val_data = df_train.iloc[train_idx], df_train.iloc[val_idx]
    x_train, y_train = train_data.drop('임신 성공 여부', axis=1), train_data["임신 성공 여부"]
    x_val, y_val = val_data.drop('임신 성공 여부', axis=1), val_data["임신 성공 여부"]

    # cat_features: 실제 존재하는 컬럼만 필터링
    # object/category 컬럼 자동 감지 + NaN 정리
    cat_cols = [c for c in x_train.columns if not pd.api.types.is_numeric_dtype(x_train[c])]
    for c in cat_cols:
        x_train[c] = x_train[c].fillna('Unknown').astype(str)
        x_val[c] = x_val[c].fillna('Unknown').astype(str)
    for c in x_train.columns:
        if c not in cat_cols:
            x_train[c] = pd.to_numeric(x_train[c], errors='coerce').fillna(-1)
            x_val[c] = pd.to_numeric(x_val[c], errors='coerce').fillna(-1)
    train_pool = Pool(x_train, label=y_train, cat_features=cat_cols)
    eval_pool = Pool(x_val, label=y_val, cat_features=cat_cols)

    # CatBoost initialization & training
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=eval_pool, verbose=100, early_stopping_rounds=250)

    # Evaluation metric
    val_pred_proba = model.predict_proba(x_val)[:, 1]
    val_pred = (val_pred_proba > 0.5).astype(int)
    add_eval_metrics(eval_metrics, y_val, val_pred, val_pred_proba)

    # Test predictions
    df_test_cp = df_test.copy()
    for c in cat_cols:
        df_test_cp[c] = df_test_cp[c].fillna('Unknown').astype(str)
    for c in df_test_cp.columns:
        if c not in cat_cols:
            df_test_cp[c] = pd.to_numeric(df_test_cp[c], errors='coerce').fillna(-1)
    test_pred_lst.append(model.predict_proba(df_test_cp)[:, 1])

1 start
0:	learn: 0.6604943	test: 0.6605394	best: 0.6605394 (0)	total: 314ms	remaining: 15m 40s
100:	learn: 0.4879471	test: 0.4877605	best: 0.4877605 (100)	total: 28.2s	remaining: 13m 29s
200:	learn: 0.4857597	test: 0.4866867	best: 0.4866849 (199)	total: 56.6s	remaining: 13m 8s
300:	learn: 0.4841273	test: 0.4864352	best: 0.4864214 (239)	total: 1m 28s	remaining: 13m 14s
400:	learn: 0.4826832	test: 0.4862224	best: 0.4862005 (397)	total: 1m 59s	remaining: 12m 55s
500:	learn: 0.4815120	test: 0.4861612	best: 0.4861110 (479)	total: 2m 32s	remaining: 12m 40s
600:	learn: 0.4803718	test: 0.4861005	best: 0.4860436 (577)	total: 3m 4s	remaining: 12m 17s
700:	learn: 0.4794423	test: 0.4860854	best: 0.4860436 (577)	total: 3m 36s	remaining: 11m 50s
800:	learn: 0.4783343	test: 0.4862117	best: 0.4860436 (577)	total: 4m 10s	remaining: 11m 27s
Stopped by overfitting detector  (250 iterations wait)

bestTest = 0.4860435555
bestIteration = 577

Shrink model to first 578 iterations.
2 start
0:	learn: 0.65583

In [24]:
print(np.array(eval_metrics['confusion']).mean(axis = 0))
print(np.array(eval_metrics['accuracy']).mean(axis = 0))
print(np.array(eval_metrics['precision']).mean(axis = 0))
print(np.array(eval_metrics['recall']).mean(axis = 0))
print(np.array(eval_metrics['f1_score']).mean(axis = 0))
print(np.array(eval_metrics['roc_auc']).mean(axis = 0))

[[7331.48  273.44]
 [2327.32  321.8 ]]
0.7463672873803062
0.5408525763373785
0.12147438478030156
0.19825340855834248
0.7407901784215482


In [25]:
from datetime import datetime

df_sub = pd.read_csv("../data/sample_submission.csv")
df_sub['probability'] = np.mean(test_pred_lst, axis = 0)

# 제출 파일 저장
df_sub.to_csv(f'OSM_{datetime.now().strftime("%m%d-%H%M")}.csv', index=False)

---
## 📋 전처리 결과 요약

| 원본 피처 수 | 전처리 후 피처 수 | 주요 추가 피처 |
|-------------|-----------------|---------------|
| 69개 | 90개+ | 비율 피처, 시기별 상호작용, 나이 기반, 불임 심각도 |

다음 단계: [03_Modeling.ipynb](./03_Modeling.ipynb)에서 CatBoost 모델을 학습한다.